# 00 Environment Sanity

Quick health checks for local development and ML dataset generation.

This notebook reads `PGURL` and `DATA_ROOT` from the environment, then runs core DB sanity queries.

In [ ]:
import os
from sqlalchemy import create_engine, text
from epilepsiae_sql_dataloader.utils import _normalize_pgurl

PGURL = os.environ.get("PGURL")
DATA_ROOT = os.environ.get("DATA_ROOT")
PYTHONNOUSERSITE = os.environ.get("PYTHONNOUSERSITE")

print(f"PGURL={PGURL}")
print(f"DATA_ROOT={DATA_ROOT}")
print(f"PYTHONNOUSERSITE={PYTHONNOUSERSITE}")

if not PGURL:
    raise RuntimeError("PGURL is not set. Run: source tools/dev/env.sh")

engine = create_engine(_normalize_pgurl(PGURL))

In [ ]:
def run_sql(label: str, sql: str):
    print("\n" + "=" * 90)
    print(label)
    print("-" * 90)
    with engine.connect() as conn:
        result = conn.execute(text(sql))
        columns = list(result.keys())
        rows = result.fetchall()

    print(" | ".join(columns))
    if not rows:
        print("(no rows)")
    else:
        for row in rows:
            print(" | ".join(str(v) for v in row))

    return rows

In [ ]:
run_sql(
    "Alembic head",
    "select max(version_num) as alembic_head from alembic_version;",
)

run_sql(
    "Core counts",
    """
    select
      (select count(*) from samples) as samples,
      (select count(*) from seizures) as seizures,
      (select count(*) from data_chunks) as data_chunks;
    """,
)

In [ ]:
run_sql(
    "seizure_state distribution",
    "select seizure_state, count(*) as rows from data_chunks group by 1 order by 1;",
)

In [ ]:
run_sql(
    "Top patients by rowcount",
    "select patient_id, count(*) as rows from data_chunks group by 1 order by rows desc limit 20;",
)

In [ ]:
run_sql(
    "Rows per partition/tableoid",
    "select tableoid::regclass as partition_name, count(*) as rows from data_chunks group by 1 order by 2 desc limit 20;",
)

## Next steps

1. Run `tools/dev/status.sh` in a terminal for the same checks in shell form.
2. Use `notebooks/01_pick_preictal_targets.ipynb` to choose candidate `(pat_id, sample_id)` pairs.
3. Follow the end-to-end pipeline in `epilepsiae_sql_dataloader/RUNBOOK.md`.